In [1]:
import polars as pl
import os
import numpy as np
import pandas as pd
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve, auc, precision_recall_fscore_support, confusion_matrix, classification_report

In [2]:
WORKING_DIR = '/group/pmc021/amunif/epi-thesis/workflow/10_Permutation Test Ranking'
DATASET_DIR = '/group/pmc021/amunif/epi-thesis/workflow/10_Permutation Test Ranking/dataset'

In [3]:
# Load dataset
MODE = 'all_features'
all_features = pl.read_parquet(os.path.join(DATASET_DIR, 'features', f'{MODE}.parquet'))
all_features

index,gene_id,all_features,value_1
u32,str,list[f64],f64
0,"""XLOC_000001""","[0.0, 0.0, … 0.0]",0.0
1,"""XLOC_000003""","[0.0, 0.0, … 0.0]",0.0
2,"""XLOC_000006""","[0.0, 0.0, … 0.0]",0.0888452
3,"""XLOC_000007""","[0.0, 0.0, … 0.0]",4.04743
4,"""XLOC_000008""","[0.0, 0.0, … 0.0]",26.7934
…,…,…,…
22149,"""XLOC_030009""","[0.0, 0.0, … 0.0]",0.0
22150,"""XLOC_030012""","[0.0, 0.0, … 0.0]",0.0
22151,"""XLOC_030014""","[0.0, 0.0, … 0.0]",0.0


In [4]:
# Convert to numpy array
all_features_np = all_features.to_numpy()
print(all_features_np)
print(all_features_np.shape)

[[0 'XLOC_000001' array([0., 0., 0., ..., 0., 0., 0.]) 0.0]
 [1 'XLOC_000003' array([0., 0., 0., ..., 0., 0., 0.]) 0.0]
 [2 'XLOC_000006' array([0., 0., 0., ..., 0., 0., 0.]) 0.0888452]
 ...
 [22151 'XLOC_030014' array([0., 0., 0., ..., 0., 0., 0.]) 0.0]
 [22152 'XLOC_030017' array([0., 0., 0., ..., 0., 0., 0.]) 0.0]
 [22153 'XLOC_030018' array([0., 0., 0., ..., 0., 0., 0.]) 0.0]]
(22154, 4)


In [5]:
# Split train, test, validation by index
data_indices = np.arange(len(all_features_np))
print(data_indices)

[    0     1     2 ... 22151 22152 22153]


In [6]:
# First split: 80% train, 20% temporary (for test + validation)
train_idx, temp_idx = train_test_split(
    data_indices, 
    test_size=0.2, 
    random_state=42  # For reproducibility
)

In [7]:
# Second split: Split temp_idx into 50% test and 50% validation
val_idx, test_idx = train_test_split(
    temp_idx, 
    test_size=0.5, 
    random_state=42  # Same random_state for consistency
)

In [8]:
print(train_idx)
print(val_idx)
print(test_idx)

[12829 12333  8225 ...  5390   860 15795]
[13202 15153  3113 ...  1158  6603 15289]
[17261  8344  1396 ...  7985   482 16493]


In [9]:
# Save train, val, and test into parquet file
train_idx_df = pd.DataFrame(train_idx, columns=['values'])
train_idx_df.to_parquet(os.path.join(DATASET_DIR, 'train_idx.parquet'))

val_idx_df = pd.DataFrame(val_idx, columns=['values'])
val_idx_df.to_parquet(os.path.join(DATASET_DIR, 'val_idx.parquet'))

test_idx_df = pd.DataFrame(test_idx, columns=['values'])
val_idx_df.to_parquet(os.path.join(DATASET_DIR, 'test_idx.parquet'))

In [10]:
train_idx_df = pd.read_parquet(os.path.join(DATASET_DIR, 'train_idx.parquet'))
len(train_idx_df["values"].to_list())

17723

In [11]:
val_idx_df = pd.read_parquet(os.path.join(DATASET_DIR, 'val_idx.parquet'))
len(val_idx_df["values"].to_list())

2215

In [12]:
test_idx_df = pd.read_parquet(os.path.join(DATASET_DIR, 'test_idx.parquet'))
len(test_idx_df["values"].to_list())

2215

In [13]:
data_df = pl.read_parquet(os.path.join(DATASET_DIR, 'gene_w_label_value_1.parquet'))
data_df

gene_id,H3K4me3,H3K4me3_count,H3K9ac,H3K9ac_count,H3K9me3,H3K9me3_count,H3K27ac,H3K27ac_count,H3K27me3,H3K27me3_count,value_1,label
str,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,f64,i32
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0888452,0
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,4.04743,1
"""XLOC_000008""","[0.0, 0.0, … 0.0]",6,"[0.0, 0.0, … 0.0]",3,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",2,"[0.0, 0.0, … 0.0]",0,26.7934,1
…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0


In [15]:
# Test re-reading the permutation list
permutation_df = pl.read_parquet(os.path.join(DATASET_DIR, 'permutation.parquet'))['markers_perm'].to_list()
permutation_df[0:10]

[['H3K4me3'],
 ['H3K9ac'],
 ['H3K9me3'],
 ['H3K27ac'],
 ['H3K27me3'],
 ['H3K4me3', 'H3K9ac'],
 ['H3K9ac', 'H3K4me3'],
 ['H3K4me3', 'H3K9me3'],
 ['H3K9me3', 'H3K4me3'],
 ['H3K4me3', 'H3K27ac']]

In [16]:
# Select only column inside the permutation
PERM_INDEX = 5
col_permute = permutation_df[PERM_INDEX]
col_permute

['H3K4me3', 'H3K9ac']

In [17]:
data_df = data_df.with_columns(histone=pl.concat_list(col_permute))
data_df

gene_id,H3K4me3,H3K4me3_count,H3K9ac,H3K9ac_count,H3K9me3,H3K9me3_count,H3K27ac,H3K27ac_count,H3K27me3,H3K27me3_count,value_1,label,histone
str,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,f64,i32,list[f64]
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0,"[0.0, 0.0, … 0.0]"
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0,"[0.0, 0.0, … 0.0]"
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0888452,0,"[0.0, 0.0, … 0.0]"
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,4.04743,1,"[0.0, 0.0, … 0.0]"
"""XLOC_000008""","[0.0, 0.0, … 0.0]",6,"[0.0, 0.0, … 0.0]",3,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",2,"[0.0, 0.0, … 0.0]",0,26.7934,1,"[0.0, 0.0, … 0.0]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0,"[0.0, 0.0, … 0.0]"
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0,"[0.0, 0.0, … 0.0]"
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0,"[0.0, 0.0, … 0.0]"


In [18]:
X = data_df["gene_id", "histone", "value_1"]
X

gene_id,histone,value_1
str,list[f64],f64
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0.0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0.0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0.0888452
"""XLOC_000007""","[0.0, 0.0, … 0.0]",4.04743
"""XLOC_000008""","[0.0, 0.0, … 0.0]",26.7934
…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0.0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0.0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0.0


In [19]:
X_np = X.to_numpy()
X_np

array([['XLOC_000001', array([0., 0., 0., ..., 0., 0., 0.]), 0.0],
       ['XLOC_000003', array([0., 0., 0., ..., 0., 0., 0.]), 0.0],
       ['XLOC_000006', array([0., 0., 0., ..., 0., 0., 0.]), 0.0888452],
       ...,
       ['XLOC_030014', array([0., 0., 0., ..., 0., 0., 0.]), 0.0],
       ['XLOC_030017', array([0., 0., 0., ..., 0., 0., 0.]), 0.0],
       ['XLOC_030018', array([0., 0., 0., ..., 0., 0., 0.]), 0.0]],
      dtype=object)

In [20]:
train_idx[0]

12829

In [21]:
class HepG2Dataset(Dataset):
    def __init__(self, total_samples, features, indexes):
        self.total_samples = total_samples
        self.features = features
        self.indexes = indexes

    def __len__(self):
        return self.total_samples

    def __getitem__(self, idx):
        r1 = random.choice(self.indexes)
        r2 = random.choice(self.indexes)
        
        feature_1 = self.features[r1, 1]
        feature_2 = self.features[r2, 1]
        feature = np.concatenate((feature_1, feature_2), axis=0)
        
        val_1 = self.features[r1, 2]
        val_2 = self.features[r2, 2]
        
        y = 1 if val_1 > val_2 else 0
        
        return feature, y

In [22]:
NUM_ITEMS = 10
BATCH_SIZE = 32

train_dataset = HepG2Dataset(int(0.8 * NUM_ITEMS), X_np, train_idx)
val_dataset = HepG2Dataset(int(0.1 * NUM_ITEMS), X_np, val_idx)
test_dataset = HepG2Dataset(int(0.1 * NUM_ITEMS), X_np, test_idx)

In [23]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

8
1
1


In [24]:
# Define the dataloader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

In [28]:
for X_batch, y_batch in train_loader:
    print(X_batch, y_batch)

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], dtype=torch.float64) tensor([1, 0, 1, 0, 0, 1, 0, 0])
